In [0]:
import  pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType, DateType,TimestampType,FloatType

catalog_name='ecommerce'

###Brands

In [0]:
df_bronze=spark.table(f"{catalog_name}.bronze.brz_brands")
df_bronze.show()

##Remove spaces from Brand_name


In [0]:
df_silver=df_bronze.withColumn('brand_name',F.trim(F.col('brand_name')))

df_silver.show()

##Remove @ in brand_code which is extra

In [0]:

df_silver = df_silver.withColumn(
    "brand_id",
    F.regexp_replace(F.col("brand_id"), r'[^A-Za-z0-9]', '')
)

df_silver.show()

##Find Distinct Category Code

In [0]:
df_silver.select("category_code").distinct().show()

In [0]:
anomalies={
    "GROCERY":"GRCY",
    "BOOKS":"BKS",
    "TOYS":"TOY"
}

df_silver=df_silver.replace(anomalies,subset=["category_code"])

df_silver.select("category_code").distinct().show()


##Load All Cleanimg of Brand Tabke in silver layer

In [0]:
df_silver.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema", "true")\
    .saveAsTable(f"{catalog_name}.silver.slv_brands")

###Category

In [0]:
df_bronze=spark.table(f"{catalog_name}.bronze.brz_category")

df_bronze.show(10)

Find Duplicates in categroy_id and Count

In [0]:
df_duplicates=df_bronze.groupBy("category_id").count().filter(F.col("count")>1)

display(df_duplicates)

Drop Duplicates Both hace same data which is incresing only Number of rows

In [0]:
df_silver=df_bronze.dropDuplicates(['category_id'])

display(df_silver)

Convert Category_code in UpperCase 

In [0]:
df_silver=df_silver.withColumn("category_id", F.upper(F.col("category_id")))

display(df_silver)

In [0]:
df_silver.write.format('delta')\
    .mode("overwrite")\
    .option("mergeSchema",True)\
    .saveAsTable(f"{catalog_name}.silver.slv_category")

###Products

Find Row and Column count

In [0]:
df_bronze=spark.read.table(f"{catalog_name}.bronze.brz_products")

row_count,column_count=df_bronze.count(),len(df_bronze.columns)

print(f"Row Count: {row_count}")
print(f"Column Count:{column_count}")

In [0]:
display(df_bronze)

Check weight_grams (contains "g")

In [0]:
df_bronze.select("weight_grams").show(5,truncate=False)

Remove "g" with blank Space " " and also convert into IntegerType

In [0]:
df_silver=df_bronze.withColumn(
    "weight_grams",
    F.regexp_replace(F.col("weight_grams"),"g"," ").cast(IntegerType())

)

df_silver.select("weight_grams").show(5,truncate=False)